# A RAG pipeline


**Download Data**



In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

# Download from the public tmdb-5000 repo
file_path = hf_hub_download(
    repo_id="alejandrowallace/tmdb-5000",
    filename="tmdb_5000_movies.csv",
    repo_type="dataset",
)

# Load into pandas and filter
df = pd.read_csv(file_path)
df = df[["original_title", "overview"]].dropna().reset_index(drop=True)

# Save locally as movies.csv for downstream tutorial steps
df.to_csv("movies.csv", index=False)

print(f"Loaded successfully! Shape: {df.shape}")
print(df.head())

Loaded successfully! Shape: (4800, 2)
                             original_title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                            overview  
0  In the 22nd century, a paraplegic Marine is di...  
1  Captain Barbossa, long believed to be dead, ha...  
2  A cryptic message from Bond’s past sends him o...  
3  Following the death of District Attorney Harve...  
4  John Carter is a war-weary, former military ca...  


In [ ]:
import nltk

nltk.download("punkt_tab")
nltk.download("punkt")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

**preprocessing and chunking**

In [ ]:

import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
df = df[['original_title', 'overview']]

# Step 1: Chunk the Overview Column using NLTKTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap = 20 )

def split_overview(overview):
    if pd.isna(overview):
        return []
    return text_splitter.split_text(str(overview))

df['chunks'] = df['overview'].apply(split_overview)

# Flatten the dataframe for easier processing
chunked_df = df.explode('chunks').reset_index(drop=True)

# Step 2: Embed with a SentenceTransformer Encoder
embedder = SentenceTransformer('all-mpnet-base-v2')  # You can choose another model

# Make sure all chunks are strings and not empty
def encode_chunk(chunk):
    if not isinstance(chunk, str) or chunk.strip() == "":
        return None
    return embedder.encode(chunk).tolist()

chunked_df['embeddings'] = chunked_df['chunks'].apply(encode_chunk)

# Drop rows where 'embeddings' is None
chunked_df.dropna(subset=['embeddings'], inplace=True)

# Step 3: Store in ChromaDB
# Initialize ChromaDB client and collection
client = chromadb.Client()
collection = client.create_collection(name='movies')

# Insert data into ChromaDB
for idx, row in chunked_df.iterrows():
    collection.add(
        ids=[str(idx)],
        embeddings=[row['embeddings']],
        metadatas=[{
            'original_title': row['original_title'],
            'chunk': row['chunks']
        }]
    )

print("Data successfully stored in ChromaDB.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Data successfully stored in ChromaDB.


**Retrieval and Answer Generation**

In [ ]:
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from chromadb import Client
from sentence_transformers import SentenceTransformer
import chromadb
import torch

# Load the SentenceTransformer model for encoding queries
sentence_model = SentenceTransformer('all-mpnet-base-v2')  # Use the same model as for embedding documents

# Load the text generation model and tokenizer
tokenizer = AutoTokenizer.from_pretrained( "Qwen/Qwen2.5-3B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
     "Qwen/Qwen2.5-3B-Instruct",
    device_map='auto' # load it in the current GPU
)
text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    return_full_text=True,
    max_new_tokens=800
)


# Function to retrieve top_k documents from ChromaDB
def retrieve_documents(query, collection, top_k=3):
    # Embed the query using the SentenceTransformer model
    query_embedding = sentence_model.encode(query).tolist()

    # Search for top_k similar documents in the collection
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    if not results['documents']:
        print("No results found for the query.")
        return [], []

    # Extract chunks and titles from the results
    chunks = []
    titles = []
    for document in results['metadatas'][0]:
        chunks.append(document['chunk'])
        titles.append(document['original_title'])

    return chunks, titles

# Function to generate answer based on retrieved chunks and titles
def generate_answer(query, chunks, titles, text_generation_pipeline):
    # Prepare the context from chunks and titles
    context = "\n\n".join([f"Title: {title}\nChunk: {chunk}" for title, chunk in zip(titles, chunks)])

    # Prepare the prompt
    prompt = f"""[INST]
    Instruction: You're an expert in movie suggestions. Your task is to analyze carefully the context and come up with an exhaustive answer to the following question:
    {query}

    Here is the context to help you:

    {context}

    [/INST]"""

    # Generate the answer using the model
    generated_text = text_generation_pipeline(prompt)[0]['generated_text']

    return generated_text

# Example usage
client = chromadb.Client()
collection = client.get_collection(name='movies')

query = "What are some good movies to watch on a rainy day?"
top_k = 3

# Retrieve documents
chunks, titles = retrieve_documents(query, collection, top_k)
print(f"Retrieved Chunks: {chunks}")
print(f"Retrieved Titles: {titles}")




Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Retrieved Chunks: ["A father is without the means to pay for his daughter's medical treatment. As a last resort, he partners with a greedy co-worker to rob a casino. When things go awry they're forced to hijack a city bus.", 'This film is a spoof along the lines of "Scary Movie" and "Not Another Teen Movie." It includes parodies of some of the indie film world\'s most renowned movies such as "Memento," "Pulp Fiction," "Magnolia," "My Big Fat Greek Wedding," "Amelie," "Run Lola Run," "El Mariachi," "The Good Girl," "Pi," "Swingers" and many others.', 'In DISASTER MOVIE, the filmmaking team behind the hits "Scary Movie," "Date Movie," "Epic Movie" and "Meet The Spartans" this time puts its unique, inimitable stamp on one of the biggest and most bloated movie genres of all time -- the disaster film.']
Retrieved Titles: ['Heist', 'My Big Fat Independent Movie', 'Disaster Movie']


**Generate answer**

In [ ]:
# Generate answer
if chunks and titles:
    answer = generate_answer(query, chunks, titles, text_generation_pipeline)
    print(answer)
else:
    print("No relevant documents found to generate an answer.")

<|im_start|>system
You are a movie recommendation assistant. Recommend movies using the provided context only. Explain briefly why each movie matches the user's request. If the context is insufficient, say so instead of inventing movie details.<|im_end|>
<|im_start|>user
Question: What are some good movies to watch on a rainy day?

Retrieved movie information:
Title: Heist
Chunk: A father is without the means to pay for his daughter's medical treatment. As a last resort, he partners with a greedy co-worker to rob a casino. When things go awry they're forced to hijack a city bus.

Title: My Big Fat Independent Movie
Chunk: This film is a spoof along the lines of "Scary Movie" and "Not Another Teen Movie." It includes parodies of some of the indie film world's most renowned movies such as "Memento," "Pulp Fiction," "Magnolia," "My Big Fat Greek Wedding," "Amelie," "Run Lola Run," "El Mariachi," "The Good Girl," "Pi," "Swingers" and many others.

Title: Disaster Movie
Chunk: In DISASTER M